## Timeline & Methodology

### 1. Data Acquisition
The key dataset we will use is the **EMS Incident Dispatch Data** from NYC Open Data. It is generated by the EMS Computer Aided Dispatch System and contains millions of historical records of medical 911 calls. Because of HIPAA (health privacy laws), the exact addresses are aggregated to a higher level of detail (like Census Tracts or Zip Codes). This works perfectly for our approach, as our algorithm will route clinics to neighborhood "centroids" (central points) rather than specific houses. We will also integrate the **NYC Hospital Locations dataset** and the **Borough Boundaries shapefiles**.

### 2. Feature Extraction
We will filter the massive EMS dataset down to the columns that matter most for calculating healthcare access and response times:
* `INCIDENT_DATETIME`: When the call was made. We will extract the hour, day of the week, and month to build our time-series forecasting model. 
* `INITIAL_CALL_TYPE` and `FINAL_CALL_TYPE`: This identifies the emergency (e.g., overdose, cardiac arrest, respiratory distress). We will filter for high-severity public health crises. 
* `INCIDENT_DISPATCH_AREA` or `ZIPCODE`: The aggregated location.
* **The Crucial Delta:** We will calculate the difference between `INCIDENT_DATETIME` (call received) and `FIRST_ON_SCENE_DATETIME` (when the ambulance arrived). This delta serves as our **Target Variable** for identifying slow-response areas.

### 3. The 3-Step AI Pipeline - Base Requirements

#### 1: Spatial Analysis (Identify the "Healthcare Deserts")
We will mathematically prove where the mobile clinics are most needed.
* We will map the locations of existing hospitals with a 0.5-mile or 1-mile radius buffer using GIS mapping tools (e.g., ArcGIS or `geopandas`). 
* We will calculate the average EMS response time (the delta mentioned above) for each census tract.
* **The Output:** An algorithmic flag identifying census tracts that have BOTH a high volume of severe medical calls AND fall outside the hospital buffer zones. These become our target "Healthcare Deserts."

####  2: Time-Series Forecasting (Predicting the Need)
Health crises fluctuate based on time and season (e.g., respiratory issues spiking on hot summer afternoons). 
* We will group the historical incident data by Census Tract and Time (e.g., 4-hour shifts).
* We will train a forecasting model (such as XGBoost or a Random Forest) using historical volume, time of day, and day of week.
* **The Output:** For a given 4-hour shift tomorrow, the model will predict how many severe incidents will occur in each "Healthcare Desert."

####  3: Dynamic Routing Optimization
With a predicted map of emergencies and a limited fleet of mobile clinics (e.g., 5 S.H.O.W. vans), we will optimize deployment.
* We will frame this as a **Facility Location Problem** or **Vehicle Routing Problem**.
* Our algorithm will take the predicted hotspots (from Step 2) and calculate the optimal intersections to park the vans so that the total driving distance to all predicted emergencies is minimized.
* **The Output:** A dynamic daily schedule for the mobile clinics, shifting their locations shift-by-shift based on predictive risk.

### 4. Dream Features

Some medical emergencies require immediate response. We propose a fleet of mobile health professionals on easily manouverable vehicles such as bicycles or motorcycles. When a 911 call is made, our text classification AI will pick up on symptoms described, and determine whether this call is a severe emergency where time is essential. This will then locate the nearest health proffessional, and target them to the location. This is essential, for cardiac arrest, the AHA rule of thumb is that for every minute without CPR or defibrillation, a patient’s chance of survival falls by 7-10%. A fast responder can start care before an ambulance arrives.